# Phase 2 Final Comparison

This notebook summarizes the saved Phase 1, Phase 2, and Stage 14 ablation results. It does not rerun experiments or alter the Phase 1 baseline.

## 1. Objective

Compare the paper-reported CAM localization result with the implemented Phase 1 CAM baseline and the Stage 13 Grad-CAM++ result. Quantify changes using actual saved metrics for the same 601 positive test images.

## 2. Phase 1 Baseline

The unchanged Phase 1 result has mean IoU 0.080087, median IoU 0.062605, IoU > 0 on 80.70%, and IoU >= 0.5 on 0.33% of the fixed test set.

## 3. Phase 2 Improvement

Stage 13 replaced only classifier-weighted CAM map generation with Grad-CAM++ using the frozen Stage 8 CNN. Threshold 0.50 and all post-processing and matching settings were retained. No box annotations were used for prediction or tuning.

## 4. Ablation Findings

Stage 14's “Grad-CAM++ removed” configuration restores the Phase 1 algorithm because map generation was the sole Stage 13 change. It therefore has the same saved metrics as the baseline. The full Phase 2 method is best by mean IoU; IoU >= 0.5 is unchanged, so the gain is not consistent across all metrics.

## 5. Final Quantitative Comparison

The reference provides mean IoU only; unavailable paper metrics are shown as N/A.

In [ ]:
import csv
import json
from pathlib import Path
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "results" / "phase2_final_results.json").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "results" / "phase2_final_results.json").is_file():
    raise FileNotFoundError("Could not locate saved Phase 2 final results")
with (ROOT / "results" / "phase2_final_results.json").open(encoding="utf-8") as stream:
    final_results = json.load(stream)
with (ROOT / "results" / "phase2_final_comparison.csv").open(newline="", encoding="utf-8") as stream:
    rows = list(csv.DictReader(stream))
header = "| Method | Mean IoU | Median IoU | IoU > 0 | IoU >= 0.5 |"
lines = [header, "|---|---:|---:|---:|---:|"]
for row in rows:
    lines.append("| " + " | ".join(row[key] for key in ("Method", "Mean IoU", "Median IoU", "IoU > 0", "IoU >= 0.5")) + " |")
display(Markdown("\n".join(lines)))

## 6. Reference Paper Comparison

The reference paper mean IoU is 0.1508. The Phase 2 mean remains below the reference; report this as a gap rather than an exact reproduction.

In [ ]:
reference = final_results["reference_paper"]["mean_iou"]
phase1 = final_results["phase1"]["mean_iou"]
phase2 = final_results["phase2"]["mean_iou"]
print(f"Phase 1 minus reference: {phase1 - reference:+.6f}")
print(f"Phase 2 minus reference: {phase2 - reference:+.6f}")
print("Phase 2 reproduces reference:", final_results["reference_comparison"]["phase2_reproduces_reference"])

## 7. Improvement Analysis

Absolute changes in IoU percentage metrics are percentage points; relative changes are percentages of the Phase 1 value.

In [ ]:
changes = final_results["improvement_over_phase1"]
print(f"Mean IoU: {changes['mean_iou']['absolute']:+.6f} ({changes['mean_iou']['relative_percent']:+.2f}%)")
print(f"Median IoU: {changes['median_iou']['absolute']:+.6f} ({changes['median_iou']['relative_percent']:+.2f}%)")
print(f"IoU > 0: {changes['percentage_iou_gt_0']['absolute_percentage_points']:+.2f} pp ({changes['percentage_iou_gt_0']['relative_percent']:+.2f}%)")
print(f"IoU >= 0.5: {changes['percentage_iou_ge_0_5']['absolute_percentage_points']:+.2f} pp ({changes['percentage_iou_ge_0_5']['relative_percent']:+.2f}%)")
print("Best ablation configuration:", final_results["ablation_summary"]["best_configuration"])

## 8. Qualitative Examples

Reuse the saved Stage 12 baseline examples and Stage 13 comparisons. The Phase 2 index deliberately includes improved, unchanged, and worsened cases.

In [ ]:
import csv
from IPython.display import Image as DisplayImage

with (ROOT / "results" / "phase2_improved_examples" / "examples.csv").open(newline="", encoding="utf-8") as stream:
    examples = list(csv.DictReader(stream))
for kind in ("improved", "unchanged", "worsened"):
    example = next((row for row in examples if row["case_type"] == kind), None)
    if example:
        print(kind, "baseline IoU", example["baseline_iou"], "Phase 2 IoU", example["improved_iou"])
        display(DisplayImage(filename=str(ROOT / example["visualization"])))
print("Existing Phase 1-only example panels:")
for path in sorted((ROOT / "results" / "cam_examples").glob("*.png"))[:2]:
    display(DisplayImage(filename=str(path)))

## 9. Conclusion

Phase 2 improved mean IoU from 0.080087 to 0.090277 (+0.010190; +12.72%) on the same fixed 601-image test set. Median IoU and IoU > 0 also improved, while IoU >= 0.5 remained unchanged at 0.33%. The Phase 2 result remains below the paper reference mean IoU of 0.1508. This is a measurable aggregate improvement over the local Phase 1 baseline, not an exact paper reproduction or a claim that every image improved.